# 08 · Learn a tiny urban world model

**Question:** Can a learned transition model support multi-step counterfactual rollouts?

**Runtime:** Python / Pyodide in JupyterLite. Choose **Run → Run All Cells**.
Each notebook is independent; start with a fresh kernel. Initial runtime/package downloads need internet.
All city geometry, residents, sensor observations and parameters are **synthetic educational fixtures**.
The longitude/latitude anchor is near Gaborone, but these are not mapped Gaborone assets or a validated city twin.

**Astra experiment:** Ask Astra to interpret one-step versus rollout error and explain why a good surrogate is not evidence of causal validity.
Run the numerical baseline first, then give Astra the exported evidence and the prompt below.
The baseline does not call or run GPT-6 Astra. No API key is needed.


In [ ]:
import sys, json, math, time
from pathlib import Path
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from twin import *
style()
print("Runtime:", sys.platform, "| data: synthetic | seed: 42")


## State + action → next state
State is dimensionless heat and traffic; actions are cooling and transit intensity.
A linear model learns the transition from synthetic simulator trajectories. Train/test splits use whole trajectories to avoid temporal leakage.
This illustrates the world-model idea of learning dynamics for imagined rollouts. It is not a pretrained visual world model or a reproduction of Ha & Schmidhuber (2018).


In [ ]:
rng=np.random.default_rng(123); states=[]; actions=[]; targets=[]; groups=[]
for episode in range(50):
    state=rng.uniform(0,2,2)
    for t in range(20):
        action=rng.uniform(0,.6,2)
        nxt=transition(state,action,rng.normal(0,.02,2))
        states.append(state.copy()); actions.append(action); targets.append(nxt); groups.append(episode)
        state=nxt
states=np.array(states); actions=np.array(actions); targets=np.array(targets); groups=np.array(groups)
X=features(states,actions); train=groups<40; test=~train
weights=np.linalg.lstsq(X[train],targets[train],rcond=None)[0]
prediction=X[test]@weights
rmse=float(np.sqrt(np.mean((prediction-targets[test])**2)))
persistence=float(np.sqrt(np.mean((states[test]-targets[test])**2)))
assert rmse<persistence
print('Held-out one-step RMSE:',rmse,'| persistence:',persistence)


In [ ]:
def rollout(action, learned=True, horizon=35):
    s=np.array([1.5,1.5]); history=[s.copy()]
    for _ in range(horizon):
        s=(features(s,action)@weights)[0] if learned else transition(s,action)
        history.append(s.copy())
    return np.array(history)
policy=np.array([.4,.5]); learned=rollout(policy); oracle=rollout(policy,False); base=rollout([0,0])
rollout_rmse=float(np.sqrt(np.mean((learned-oracle)**2)))
fig,axes=plt.subplots(1,2,figsize=(12,4))
for i,ax in enumerate(axes):
    ax.plot(base[:,i],label='Learned baseline'); ax.plot(learned[:,i],label='Learned intervention'); ax.plot(oracle[:,i],'--',label='Simulator intervention')
    ax.set(xlabel='Step',ylabel='Dimensionless state',title=['Heat','Traffic'][i]); ax.legend()
plt.tight_layout(); plt.show()
outside=np.array([1.2,1.2]); print('Out-of-training-support action:',outside,'- no validated forecast')
assert (outside>actions.max(axis=0)).all()
export_json('08_world_model.json',dict(synthetic=True,one_step_rmse=rmse,persistence_rmse=persistence,rollout_rmse=rollout_rmse,training_action_max=.6))


## Extend with Codex or Astra
Replace linear dynamics with nonlinear congestion and test where the surrogate fails. Add ensembles, uncertainty calibration and action-support checks before any policy optimizer. Explore a separately hosted visual world model only after defining geography, scale and evaluation truth.

Keep the baseline seed and tests fixed while changing one assumption. Record the model name,
date, exact prompt, response and error metrics. Generated code must pass the same checks.
Download files in `exports/` and your edited notebook; browser storage does not commit changes to GitHub.
See [the project guide](https://github.com/jltobias/JupyterLite-GPT-6-Astra-Geospatial/blob/main/docs/GUIDE.md)
for capabilities, evidence, sources and the route to real data.
